# 02 — Four model definitions for vertebra / lumbar-level heatmap localization

This notebook defines exactly the four model families intended for the localization experiments:

1. `ConditionalUNet`
2. `ConditionalAttentionUNet`
3. `ConditionalUNETR`
4. `ConditionalSwinUNet`

All models accept:

```python
logits = model(image, level_idx)
```

where `image` is `[B, C, H, W]` and `level_idx` is `[B]`.

In [ ]:
import math
from typing import Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    LEVELS
except NameError:
    LEVELS = ["L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1"]

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, mid_ch: Optional[int] = None, dropout: float = 0.0):
        super().__init__()
        if mid_ch is None:
            mid_ch = out_ch
        layers = [
            nn.Conv2d(in_ch, mid_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_ch),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        layers += [
            nn.Conv2d(mid_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class Down(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, dropout: float = 0.0):
        super().__init__()
        self.net = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_ch, out_ch, dropout=dropout))

    def forward(self, x):
        return self.net(x)


class Up(nn.Module):
    def __init__(self, in_ch: int, skip_ch: int, out_ch: int, dropout: float = 0.0):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.conv = DoubleConv(out_ch + skip_ch, out_ch, dropout=dropout)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([skip, x], dim=1))


class LevelFiLM(nn.Module):
    """Feature-wise linear modulation from target-level metadata."""
    def __init__(self, n_levels: int, channels: int, emb_dim: int = 32):
        super().__init__()
        self.embedding = nn.Embedding(n_levels, emb_dim)
        self.to_gamma_beta = nn.Sequential(
            nn.Linear(emb_dim, emb_dim),
            nn.ReLU(inplace=True),
            nn.Linear(emb_dim, channels * 2),
        )
        nn.init.zeros_(self.to_gamma_beta[-1].weight)
        nn.init.zeros_(self.to_gamma_beta[-1].bias)

    def forward(self, x, level_idx):
        gb = self.to_gamma_beta(self.embedding(level_idx))
        gamma, beta = gb.chunk(2, dim=1)
        gamma = gamma[:, :, None, None]
        beta = beta[:, :, None, None]
        return x * (1.0 + gamma) + beta


class AttentionGate(nn.Module):
    """Attention gate matching the user's Keras-style block.

    x      : skip-connection tensor from the encoder, usually [B, Fx, Hx, Wx]
    gating : deeper gating tensor from the next-lowest layer, usually [B, Fg, Hx/2, Wx/2]

    Implementation mirrors:
    theta_x = Conv1x1(stride=2)(x)
    phi_g   = Conv1x1(stride=1)(gating)
    psi     = sigmoid(Conv1x1(relu(theta_x + phi_g)))
    psi     = upsample to x size
    result  = BatchNorm(Conv1x1(psi * x))
    """
    def __init__(self, x_ch: int, gating_ch: int, inter_ch: int):
        super().__init__()
        self.theta_x = nn.Conv2d(x_ch, inter_ch, kernel_size=1, stride=2, padding=0, bias=False)
        self.phi_g = nn.Conv2d(gating_ch, inter_ch, kernel_size=1, stride=1, padding=0, bias=True)
        self.psi = nn.Conv2d(inter_ch, 1, kernel_size=1, stride=1, padding=0, bias=True)
        self.result = nn.Sequential(
            nn.Conv2d(x_ch, x_ch, kernel_size=1, padding=0, bias=False),
            nn.BatchNorm2d(x_ch),
        )

    def forward(self, x, gating):
        theta_x = self.theta_x(x)
        phi_g = self.phi_g(gating)
        if phi_g.shape[-2:] != theta_x.shape[-2:]:
            phi_g = F.interpolate(phi_g, size=theta_x.shape[-2:], mode="bilinear", align_corners=False)
        attn = F.relu(theta_x + phi_g, inplace=True)
        attn = torch.sigmoid(self.psi(attn))
        attn = F.interpolate(attn, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return self.result(x * attn)

# U-Net

In [ ]:
class ConditionalUNet(nn.Module):
    """Classic 2D U-Net with target-level conditioning and one heatmap output."""
    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        n_levels: int = 5,
        base_c: int = 32,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, base_c, dropout=dropout)
        self.down1 = Down(base_c, base_c * 2, dropout=dropout)
        self.down2 = Down(base_c * 2, base_c * 4, dropout=dropout)
        self.down3 = Down(base_c * 4, base_c * 8, dropout=dropout)
        self.down4 = Down(base_c * 8, base_c * 16, dropout=dropout)
        self.film = LevelFiLM(n_levels, base_c * 16)
        self.up1 = Up(base_c * 16, base_c * 8, base_c * 8, dropout=dropout)
        self.up2 = Up(base_c * 8, base_c * 4, base_c * 4, dropout=dropout)
        self.up3 = Up(base_c * 4, base_c * 2, base_c * 2, dropout=dropout)
        self.up4 = Up(base_c * 2, base_c, base_c, dropout=dropout)
        self.outc = nn.Conv2d(base_c, out_channels, kernel_size=1)

    def forward(self, x, level_idx):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.film(x5, level_idx)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# Attention U-Net

In [ ]:
class ConditionalAttentionUNet(nn.Module):
    """Attention U-Net using the requested attention gate formulation."""
    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        n_levels: int = 5,
        base_c: int = 32,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, base_c, dropout=dropout)
        self.down1 = Down(base_c, base_c * 2, dropout=dropout)
        self.down2 = Down(base_c * 2, base_c * 4, dropout=dropout)
        self.down3 = Down(base_c * 4, base_c * 8, dropout=dropout)
        self.down4 = Down(base_c * 8, base_c * 16, dropout=dropout)
        self.film = LevelFiLM(n_levels, base_c * 16)

        self.att4 = AttentionGate(x_ch=base_c * 8, gating_ch=base_c * 16, inter_ch=base_c * 8)
        self.att3 = AttentionGate(x_ch=base_c * 4, gating_ch=base_c * 8, inter_ch=base_c * 4)
        self.att2 = AttentionGate(x_ch=base_c * 2, gating_ch=base_c * 4, inter_ch=base_c * 2)
        self.att1 = AttentionGate(x_ch=base_c, gating_ch=base_c * 2, inter_ch=base_c)

        self.up1 = Up(base_c * 16, base_c * 8, base_c * 8, dropout=dropout)
        self.up2 = Up(base_c * 8, base_c * 4, base_c * 4, dropout=dropout)
        self.up3 = Up(base_c * 4, base_c * 2, base_c * 2, dropout=dropout)
        self.up4 = Up(base_c * 2, base_c, base_c, dropout=dropout)
        self.outc = nn.Conv2d(base_c, out_channels, kernel_size=1)

    def forward(self, x, level_idx):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.film(x5, level_idx)

        x4_att = self.att4(x4, x5)
        d4 = self.up1(x5, x4_att)
        x3_att = self.att3(x3, d4)
        d3 = self.up2(d4, x3_att)
        x2_att = self.att2(x2, d3)
        d2 = self.up3(d3, x2_att)
        x1_att = self.att1(x1, d2)
        d1 = self.up4(d2, x1_att)
        return self.outc(d1)

In [ ]:
class AttentionGate(nn.Module):
    """Attention gate for Attention U-Net.

    This version starts nearly identity-like, so skip connections are not
    strongly suppressed at the beginning of training.
    """
    def __init__(self, x_ch: int, gating_ch: int, inter_ch: int):
        super().__init__()

        self.theta_x = nn.Conv2d(
            x_ch,
            inter_ch,
            kernel_size=1,
            stride=2,
            padding=0,
            bias=False,
        )

        self.phi_g = nn.Conv2d(
            gating_ch,
            inter_ch,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=True,
        )

        self.psi = nn.Conv2d(
            inter_ch,
            1,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=True,
        )

        self.result = nn.Sequential(
            nn.Conv2d(x_ch, x_ch, kernel_size=1, padding=0, bias=False),
            nn.BatchNorm2d(x_ch),
        )

        # Important:
        # sigmoid(2.0) ≈ 0.88, so the gate initially keeps most skip information.
        nn.init.constant_(self.psi.bias, 2.0)

    def forward(self, x, gating):
        theta_x = self.theta_x(x)
        phi_g = self.phi_g(gating)

        if phi_g.shape[-2:] != theta_x.shape[-2:]:
            phi_g = F.interpolate(
                phi_g,
                size=theta_x.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

        attn = F.relu(theta_x + phi_g, inplace=True)
        attn = torch.sigmoid(self.psi(attn))

        attn = F.interpolate(
            attn,
            size=x.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        return self.result(x * attn)

In [ ]:
class ConditionalAttentionUNet(nn.Module):
    """Attention U-Net with level conditioning at every encoder scale.

    The important change compared with the previous version:
    LevelFiLM is applied not only at the bottleneck, but also to all skip levels.
    """
    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        n_levels: int = 5,
        base_c: int = 32,
        dropout: float = 0.0,
    ):
        super().__init__()

        # Encoder
        self.inc = DoubleConv(in_channels, base_c, dropout=dropout)              # H
        self.down1 = Down(base_c, base_c * 2, dropout=dropout)                   # H/2
        self.down2 = Down(base_c * 2, base_c * 4, dropout=dropout)               # H/4
        self.down3 = Down(base_c * 4, base_c * 8, dropout=dropout)               # H/8
        self.down4 = Down(base_c * 8, base_c * 16, dropout=dropout)              # H/16

        # Level conditioning at all scales
        self.film1 = LevelFiLM(n_levels, base_c)
        self.film2 = LevelFiLM(n_levels, base_c * 2)
        self.film3 = LevelFiLM(n_levels, base_c * 4)
        self.film4 = LevelFiLM(n_levels, base_c * 8)
        self.film5 = LevelFiLM(n_levels, base_c * 16)

        # Attention gates
        self.att4 = AttentionGate(
            x_ch=base_c * 8,
            gating_ch=base_c * 16,
            inter_ch=base_c * 8,
        )

        self.att3 = AttentionGate(
            x_ch=base_c * 4,
            gating_ch=base_c * 8,
            inter_ch=base_c * 4,
        )

        self.att2 = AttentionGate(
            x_ch=base_c * 2,
            gating_ch=base_c * 4,
            inter_ch=base_c * 2,
        )

        self.att1 = AttentionGate(
            x_ch=base_c,
            gating_ch=base_c * 2,
            inter_ch=base_c,
        )

        # Decoder
        self.up1 = Up(base_c * 16, base_c * 8, base_c * 8, dropout=dropout)      # H/8
        self.up2 = Up(base_c * 8, base_c * 4, base_c * 4, dropout=dropout)       # H/4
        self.up3 = Up(base_c * 4, base_c * 2, base_c * 2, dropout=dropout)       # H/2
        self.up4 = Up(base_c * 2, base_c, base_c, dropout=dropout)              # H

        self.outc = nn.Conv2d(base_c, out_channels, kernel_size=1)

    def forward(self, x, level_idx):
        # Encoder + level conditioning
        x1 = self.inc(x)
        x1 = self.film1(x1, level_idx)

        x2 = self.down1(x1)
        x2 = self.film2(x2, level_idx)

        x3 = self.down2(x2)
        x3 = self.film3(x3, level_idx)

        x4 = self.down3(x3)
        x4 = self.film4(x4, level_idx)

        x5 = self.down4(x4)
        x5 = self.film5(x5, level_idx)

        # Decoder with attention-gated skips
        x4_att = self.att4(x4, x5)
        d4 = self.up1(x5, x4_att)

        x3_att = self.att3(x3, d4)
        d3 = self.up2(d4, x3_att)

        x2_att = self.att2(x2, d3)
        d2 = self.up3(d3, x2_att)

        x1_att = self.att1(x1, d2)
        d1 = self.up4(d2, x1_att)

        return self.outc(d1)

# UNETR

In [ ]:
class ConditionalUNETR(nn.Module):
    """Compact 2D UNETR-style model for heatmap localization.

    2.5D input -> ViT bottleneck at H/16 -> CNN decoder with proper
    H/8, H/4, H/2, H skip levels.
    """
    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        n_levels: int = 5,
        img_size: int = 256,
        patch_size: int = 16,
        embed_dim: int = 256,
        depth: int = 6,
        num_heads: int = 8,
        base_c: int = 32,
        dropout: float = 0.0,
    ):
        super().__init__()
        assert img_size % patch_size == 0, "img_size must be divisible by patch_size"

        self.img_size = img_size
        self.patch_size = patch_size
        self.grid = img_size // patch_size

        # CNN skip path
        self.stem1 = DoubleConv(in_channels, base_c, dropout=dropout)             # H
        self.stem2 = Down(base_c, base_c * 2, dropout=dropout)                    # H/2
        self.stem3 = Down(base_c * 2, base_c * 4, dropout=dropout)                # H/4
        self.stem4 = Down(base_c * 4, base_c * 8, dropout=dropout)                # H/8

        # ViT bottleneck at H/16
        self.patch_embed = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )

        self.pos_embed = nn.Parameter(
            torch.zeros(1, self.grid * self.grid, embed_dim)
        )

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=depth)

        # Level conditioning at bottleneck
        self.film = LevelFiLM(n_levels, embed_dim)

        # Project transformer feature to decoder channels
        self.proj = DoubleConv(embed_dim, base_c * 16, dropout=dropout)            # H/16

        # Proper decoder: H/16 -> H/8 -> H/4 -> H/2 -> H
        self.up1 = Up(base_c * 16, base_c * 8, base_c * 8, dropout=dropout)        # H/8
        self.up2 = Up(base_c * 8, base_c * 4, base_c * 4, dropout=dropout)         # H/4
        self.up3 = Up(base_c * 4, base_c * 2, base_c * 2, dropout=dropout)         # H/2
        self.up4 = Up(base_c * 2, base_c, base_c, dropout=dropout)                # H

        self.outc = nn.Conv2d(base_c, out_channels, kernel_size=1)

        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x, level_idx):
        s1 = self.stem1(x)      # H
        s2 = self.stem2(s1)     # H/2
        s3 = self.stem3(s2)     # H/4
        s4 = self.stem4(s3)     # H/8

        z = self.patch_embed(x) # H/16
        b, c, h, w = z.shape

        tokens = z.flatten(2).transpose(1, 2)
        tokens = tokens + self.pos_embed[:, : h * w, :]

        tokens = self.transformer(tokens)

        z = tokens.transpose(1, 2).reshape(b, c, h, w)
        z = self.film(z, level_idx)
        z = self.proj(z)

        d = self.up1(z, s4)     # H/8
        d = self.up2(d, s3)     # H/4
        d = self.up3(d, s2)     # H/2
        d = self.up4(d, s1)     # H

        return self.outc(d)

# Swin U-Net

In [ ]:
class WindowAttention(nn.Module):
    def __init__(self, dim: int, num_heads: int = 4, window_size: int = 8, dropout: float = 0.0):
        super().__init__()
        assert dim % num_heads == 0
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.scale = (dim // num_heads) ** -0.5
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        # x: B,H,W,C
        b, h, w, c = x.shape
        ws = self.window_size
        pad_h = (ws - h % ws) % ws
        pad_w = (ws - w % ws) % ws
        if pad_h or pad_w:
            x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h))
        hp, wp = x.shape[1], x.shape[2]

        xw = x.view(b, hp // ws, ws, wp // ws, ws, c).permute(0, 1, 3, 2, 4, 5).reshape(-1, ws * ws, c)
        qkv = self.qkv(xw).reshape(xw.shape[0], ws * ws, 3, self.num_heads, c // self.num_heads)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(xw.shape[0], ws * ws, c)
        out = self.proj(out)
        out = self.drop(out)
        out = out.view(b, hp // ws, wp // ws, ws, ws, c).permute(0, 1, 3, 2, 4, 5).reshape(b, hp, wp, c)
        return out[:, :h, :w, :]


class SwinBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int = 4, window_size: int = 8, shift: bool = False, dropout: float = 0.0):
        super().__init__()
        self.shift = shift
        self.window_size = window_size
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, num_heads=num_heads, window_size=window_size, dropout=dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # x: B,C,H,W
        x = x.permute(0, 2, 3, 1)
        shortcut = x
        y = self.norm1(x)
        if self.shift:
            shift_size = self.window_size // 2
            y = torch.roll(y, shifts=(-shift_size, -shift_size), dims=(1, 2))
        y = self.attn(y)
        if self.shift:
            shift_size = self.window_size // 2
            y = torch.roll(y, shifts=(shift_size, shift_size), dims=(1, 2))
        x = shortcut + y
        x = x + self.mlp(self.norm2(x))
        return x.permute(0, 3, 1, 2).contiguous()


class SwinStage(nn.Module):
    def __init__(self, dim: int, num_heads: int, window_size: int, dropout: float = 0.0):
        super().__init__()
        self.blocks = nn.Sequential(
            SwinBlock(dim, num_heads=num_heads, window_size=window_size, shift=False, dropout=dropout),
            SwinBlock(dim, num_heads=num_heads, window_size=window_size, shift=True, dropout=dropout),
        )

    def forward(self, x):
        return self.blocks(x)


class PatchDown(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, out_ch, kernel_size=2, stride=2)

    def forward(self, x):
        return self.proj(x)


class ConditionalSwinUNet(nn.Module):
    """Compact Swin-Unet-like model for 2.5D heatmap localization.

    It uses local window self-attention blocks with alternating shifted windows, U-Net-like skip
    connections, and target-level FiLM conditioning at the bottleneck.
    """
    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        n_levels: int = 5,
        base_c: int = 32,
        window_size: int = 8,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.patch = nn.Conv2d(in_channels, base_c, kernel_size=4, stride=4, padding=0)
        self.stage1 = SwinStage(base_c, num_heads=4, window_size=window_size, dropout=dropout)
        self.down1 = PatchDown(base_c, base_c * 2)
        self.stage2 = SwinStage(base_c * 2, num_heads=4, window_size=window_size, dropout=dropout)
        self.down2 = PatchDown(base_c * 2, base_c * 4)
        self.stage3 = SwinStage(base_c * 4, num_heads=8, window_size=window_size, dropout=dropout)
        self.down3 = PatchDown(base_c * 4, base_c * 8)
        self.bottleneck = SwinStage(base_c * 8, num_heads=8, window_size=window_size, dropout=dropout)
        self.film = LevelFiLM(n_levels, base_c * 8)

        self.up1 = Up(base_c * 8, base_c * 4, base_c * 4, dropout=dropout)
        self.up2 = Up(base_c * 4, base_c * 2, base_c * 2, dropout=dropout)
        self.up3 = Up(base_c * 2, base_c, base_c, dropout=dropout)
        self.final_up = nn.Sequential(
            nn.ConvTranspose2d(base_c, base_c, kernel_size=4, stride=4),
            DoubleConv(base_c, base_c, dropout=dropout),
        )
        self.outc = nn.Conv2d(base_c, out_channels, kernel_size=1)

    def forward(self, x, level_idx):
        input_size = x.shape[-2:]
        s1 = self.stage1(self.patch(x))
        s2 = self.stage2(self.down1(s1))
        s3 = self.stage3(self.down2(s2))
        b = self.bottleneck(self.down3(s3))
        b = self.film(b, level_idx)

        d = self.up1(b, s3)
        d = self.up2(d, s2)
        d = self.up3(d, s1)
        d = self.final_up(d)
        if d.shape[-2:] != input_size:
            d = F.interpolate(d, size=input_size, mode="bilinear", align_corners=False)
        return self.outc(d)

In [ ]:
def get_localization_model(
    model_name: str,
    in_channels: int = 3,
    out_channels: int = 1,
    n_levels: int = 5,
    base_c: int = 32,
    dropout: float = 0.0,
    img_size: int = 256,
):
    name = model_name.lower()
    if name == "unet":
        return ConditionalUNet(in_channels, out_channels, n_levels, base_c, dropout)
    if name in {"attention_unet", "att_unet"}:
        return ConditionalAttentionUNet(in_channels, out_channels, n_levels, base_c, dropout)
    if name == "unetr":
        return ConditionalUNETR(in_channels=in_channels, out_channels=out_channels, n_levels=n_levels, img_size=img_size, base_c=base_c, dropout=dropout)
    if name in {"swin_unet", "swinunet"}:
        return ConditionalSwinUNet(in_channels=in_channels, out_channels=out_channels, n_levels=n_levels, base_c=base_c, dropout=dropout)
    raise ValueError("model_name must be one of: 'unet', 'attention_unet', 'unetr', 'swin_unet'")


def count_parameters(model: nn.Module, trainable_only: bool = True) -> int:
    params = model.parameters()
    if trainable_only:
        params = [p for p in params if p.requires_grad]
    return sum(p.numel() for p in params)

## 3D V-Net

In [ ]:
class ResidualConvBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, n_convs=2, kernel_size=3, dropout=0.0):
        super().__init__()
        padding = kernel_size // 2
        layers = []
        ch = in_ch
        for _ in range(n_convs):
            layers += [
                nn.Conv3d(ch, out_ch, kernel_size=kernel_size, padding=padding, bias=False),
                nn.InstanceNorm3d(out_ch, affine=True),
                nn.PReLU(out_ch),
            ]
            if dropout > 0:
                layers.append(nn.Dropout3d(dropout))
            ch = out_ch
        self.net = nn.Sequential(*layers)
        self.proj = nn.Identity() if in_ch == out_ch else nn.Conv3d(in_ch, out_ch, kernel_size=1, bias=False)
        self.act = nn.PReLU(out_ch)

    def forward(self, x):
        return self.act(self.net(x) + self.proj(x))


class DownBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, n_convs=2):
        super().__init__()
        self.down = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel_size=2, stride=2, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.PReLU(out_ch),
        )
        self.block = ResidualConvBlock3D(out_ch, out_ch, n_convs=n_convs)

    def forward(self, x):
        return self.block(self.down(x))


class UpBlock3D(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, n_convs=2):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose3d(in_ch, out_ch, kernel_size=2, stride=2, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.PReLU(out_ch),
        )
        self.skip_proj = nn.Identity() if skip_ch == out_ch else nn.Conv3d(skip_ch, out_ch, kernel_size=1, bias=False)
        self.block = ResidualConvBlock3D(out_ch, out_ch, n_convs=n_convs)

    def forward(self, x, skip):
        x = self.up(x)
        # Protect against odd-size rounding if you change fixed_depth/img_size.
        if x.shape[-3:] != skip.shape[-3:]:
            x = F.interpolate(x, size=skip.shape[-3:], mode='trilinear', align_corners=False)
        return self.block(x + self.skip_proj(skip))


class VNet3DLocalization(nn.Module):
    def __init__(self, in_channels=1, out_channels=5, base_channels=8):
        super().__init__()
        c = base_channels
        self.enc1 = ResidualConvBlock3D(in_channels, c, n_convs=1)
        self.enc2 = DownBlock3D(c, c * 2, n_convs=2)
        self.enc3 = DownBlock3D(c * 2, c * 4, n_convs=2)
        self.enc4 = DownBlock3D(c * 4, c * 8, n_convs=2)
        self.bottleneck = DownBlock3D(c * 8, c * 16, n_convs=2)

        self.dec4 = UpBlock3D(c * 16, c * 8, c * 8, n_convs=2)
        self.dec3 = UpBlock3D(c * 8, c * 4, c * 4, n_convs=2)
        self.dec2 = UpBlock3D(c * 4, c * 2, c * 2, n_convs=2)
        self.dec1 = UpBlock3D(c * 2, c, c, n_convs=1)
        self.out = nn.Conv3d(c, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        b = self.bottleneck(e4)
        d4 = self.dec4(b, e4)
        d3 = self.dec3(d4, e3)
        d2 = self.dec2(d3, e2)
        d1 = self.dec1(d2, e1)
        return self.out(d1)
